In [ ]:
import os
from pathlib import Path 

import tqdm
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from benchmark_data_leakage.chembl_requester import ChEMBLRequester
from benchmark_data_leakage.utils import find_repo_root, invert_inequality, calc_neglog10_val

repo_root = find_repo_root()
data_dir = repo_root / "data"
out_dir = data_dir / "out"
fig_dir = repo_root / "figures"
fig_dir.mkdir(exist_ok=True)

### Load the data we will use in the analyses

In [ ]:
# Get all ChEMBL activity data
# Add your chembl database connection details here
chembl_req = ChEMBLRequester(
    host=
    user=
    password= # Don't commit (only used for ChEMBL anyway, so doesn't really matter)
    dbname= # ChEMBL 36 used here, but earlier versions are likely fine (the papers are old!)
)

chembl_protein_targets = chembl_req.get_single_protein_targets()
chembl_protein_targets = pd.DataFrame(chembl_protein_targets)

# Querying all activity data. This consumes ~12 GB of RAM. constrain to only your 
activity_data = chembl_req.get_all_single_protein_activity_data(
    # Constraining by target_chembl_id's is optional.
    # None = No constraint, this uses 12GB of data and takes a few minutes
    # ['CHEMBL123', 'CHEMBL456'] = only query those targets
    target_chembl_ids=None,
)
activity_data = pd.DataFrame(activity_data)

# Join some extra columns we want later
data_split = pd.read_csv(out_dir / "FEP_data_split.csv").set_index("uniprot_id")["data_split"]
chembl_protein_targets = chembl_protein_targets.join(
    data_split,
    on="uniprot_id"
)
activity_data = activity_data.join(
    chembl_protein_targets.set_index("chembl_target_id")[["gene_name", "organism", "data_split"]],
    on="chembl_target_id"
)

# Load the FEPp benchmark data
benchmark_data = pd.read_csv(out_dir / "FEPp_benchmark.csv")

### Analyse how often ligands in the benchmark appears in test and train/val

In [ ]:
gene_name = "MAPK8"   # CDK2, TYK2, MAPK8, MAPK14
benchmark_slice = benchmark_data.loc[
    benchmark_data["gene_name"] == gene_name
]

n_ligands = len(benchmark_slice)
n_ligands_with_chembl_ids = (~benchmark_slice["chembl_ligand_id"].isna()).sum()
ligand_ids = benchmark_slice["chembl_ligand_id"].values

In [ ]:
print(f"{gene_name} appears with {n_ligands} compounds in the FEP+ 4 test set")
print(f"Of those {n_ligands_with_chembl_ids} compounds exist in ChEMBL")
print(f"In ChEMBL they appear against different targets this many times:")
activity_data.loc[
    activity_data["chembl_ligand_id"].isin(ligand_ids)
].drop_duplicates(
    ["gene_name", "chembl_ligand_id"]
).value_counts(["data_split", "gene_name", "organism"]).iloc[:20]

### Create figures and tables for slides (JNK1 / MAPK8)

In [ ]:
df_bench_slice = benchmark_data.loc[benchmark_data["gene_name"] == "MAPK8"].copy()
def get_pIC50(row: pd.Series) -> float:
    if row["measurement_unit"] == "uM":
        return -np.log10(row["measurement_value"]) + 6
    elif row["measurement_unit"] == "nM":
        return -np.log10(row["measurement_value"]) + 9
df_bench_slice["pIC50"] = df_bench_slice.apply(get_pIC50, axis=1)
df_bench_slice = df_bench_slice[["name", "pIC50", "chembl_ligand_id", "chembl_target_id"]].rename(columns={
    "pIC50": "FEP+_pIC50",
    "name": "FEP+_ligand_name"
})

focus_assays = ["CHEMBL860181", "CHEMBL860184"]
activity_data_slice = activity_data.loc[
    activity_data["chembl_assay_id"].isin(focus_assays)
    & (activity_data["measurement_type"] == "IC50")
    & (activity_data["chembl_ligand_id"].isin(df_bench_slice["chembl_ligand_id"].values))
].copy()
# Fill missing pchembl_value
activity_data_slice["pchembl_value"] = activity_data_slice["pchembl_value"].fillna(
    activity_data_slice.apply(calc_neglog10_val, axis=1)
).astype(float)
# Creating relation for pIC50 values (e.g. invert the normal relation)
activity_data_slice["pval_relation"] = activity_data_slice["measurement_relation"].apply(invert_inequality)

df_combined = activity_data_slice.join(
    df_bench_slice.set_index("chembl_ligand_id")["FEP+_ligand_name"],
    on="chembl_ligand_id"
).join(
    df_bench_slice.set_index(["chembl_ligand_id", "chembl_target_id"])["FEP+_pIC50"],
    on=["chembl_ligand_id", "chembl_target_id"]
)
def include_rel_in_val(row: pd.Series) -> str:
    if row["pval_relation"] == "=":
        return f'{row["pchembl_value"]:.2f}'
    elif row["pval_relation"] == "<":
        return f'<{row["pchembl_value"]:.2f}'
    raise RuntimeError

df_combined["pIC50 (str)"] = df_combined.apply(include_rel_in_val, axis=1)

In [ ]:
# 1. Set the visual style and scaling for a slide/paper
import scipy
import seaborn as sns
import matplotlib.pyplot as plt


df = df_combined.pivot(
    columns=["gene_name", "chembl_assay_id"],
    values="pchembl_value",
    index=["FEP+_ligand_name", "chembl_ligand_id"],
    )
x_col = ("MAPK8", "CHEMBL860181")
y_col = ("MAPK9", "CHEMBL860184")
x_label = "Test set: JNK1 [pIC50]"
y_label = "Train set: JNK2 [pIC50]"

jnk1_vals = df[x_col]
jnk2_vals = df[y_col]

corr, _ = scipy.stats.pearsonr(jnk1_vals, jnk2_vals)
print(corr)

title = f"Data points in both splits ($r$={corr:.2f})"
sns.set_theme(style="ticks", context="talk") # "talk" scales fonts for slides

# 2. Initialize the figure with a square aspect ratio
fig, ax = plt.subplots(figsize=(6, 6))

# 3. Create the scatter plot
sns.scatterplot(
    data=df,
    x=x_col,
    y=y_col,
    s=80,                # Larger points
    edgecolor="w",       # White outline for clarity
    alpha=0.8,           # Slight transparency for overlaps
    color="#2d5883",     # Professional dark blue/grey
    ax=ax
)

# 4. Set custom labels
ax.set_xlabel(x_label, fontweight='bold', fontsize=14)
ax.set_ylabel(y_label, fontweight='bold', fontsize=14)
if title:
    ax.set_title(title, fontsize=16, pad=15)

# Add a subtle dashed grid
ax.grid(
    visible=True, 
    linestyle='--', 
    linewidth=1.0, 
    color='lightgray', 
    alpha=1.0
)

# 5. Add a Unity Line (y=x) to show relative affinity
# lims = [
#     np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
#     np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
# ]
# ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0, label="Unity")
# ax.set_xlim(lims)
# ax.set_ylim(lims)

# 6. Final aesthetic cleanup
ax.set_aspect('equal')
sns.despine() # Remove top and right spines
plt.tight_layout()
plt.savefig(fig_dir / "jnk1_jnk2_leakage_plot.png", dpi=300, bbox_inches='tight', transparent=False)

In [ ]:
# Create data for table
df_for_table = df_combined.pivot(
    columns=["gene_name", "chembl_assay_id"],
    values="pIC50 (str)",
    index=["FEP+_ligand_name", "chembl_ligand_id"],
    ).sort_values(("MAPK8", "CHEMBL860181"))
df_for_table

In [ ]:
# Convert to latex rows

def pic50_to_rgb(value: float, vmin: float = 5.0, vmax: float = 8.0) -> str:
    norm = np.clip((value - vmin) / (vmax - vmin), 0, 1)
    cmap = plt.get_cmap('RdYlGn')
    r, g, b, _ = cmap(norm)
    return f"{r:.2f},{g:.2f},{b:.2f}"


def parse_pic50(value: str) -> float:
    """Handle <5.0 and normal float strings."""
    if isinstance(value, str) and value.startswith('<'):
        return float(value[1:])
    return float(value)


def df_to_latex_rows(df) -> str:
    rows = []
    for (lig_name, chembl_id), row in df.iterrows():
        cells = [lig_name.replace("_", "\\_"), chembl_id]
        for col in df.columns:
            raw = row[col]
            numeric = parse_pic50(raw)
            color = pic50_to_rgb(numeric)
            cells.append(f"\\cellcolor[rgb]{{{color}}} {raw}")
        rows.append(" & ".join(cells) + " \\\\")
    return "\n".join(rows)

print(df_to_latex_rows(df_for_table))